# Notebook 04 â€” Privacy Filter (Face + Text Detection)

Pretrained models â€” **no training needed**. Sets up, tests, and exports to ONNX for Jetson.

## Cell 04-01 | Mount & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess, torch, cv2
import numpy as np
from PIL import Image

BASE    = '/content/drive/MyDrive/ARGUS'
MODELS  = f'{BASE}/models'
EXPORTS = f'{BASE}/exports'
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'insightface', 'onnxruntime-gpu', 'opencv-python-headless'])
print(f'Ready | Device: {DEVICE}')

## Cell 04-02 | Download SCRFD Face Detector

In [ ]:
import subprocess, os, zipfile, tarfile as _tarfile

def _safe_dl(url, dest, min_mb=None):
    """wget -c resume. If file exists but too small, delete and restart."""
    if os.path.exists(dest):
        mb = os.path.getsize(dest) / 1e6
        if min_mb and mb < min_mb * 0.90:
            print(f"  Partial file {os.path.basename(dest)} ({mb:.0f}/{min_mb} MB) — removing")
            os.remove(dest)
        else:
            print(f"  ✓ {os.path.basename(dest)} ({mb:.0f} MB)")
            return True
    print(f"  Downloading {os.path.basename(dest)} ...")
    r = subprocess.run(["wget", "-c", "--show-progress", "-O", dest, url])
    return r.returncode == 0

def _verify(path):
    """Return True if zip/tar is intact, delete it and return False if corrupt."""
    try:
        if path.endswith(".zip"):
            with zipfile.ZipFile(path) as z:
                bad = z.testzip()
                if bad: raise ValueError(bad)
        elif path.endswith((".tar.gz", ".tgz", ".tar")):
            with _tarfile.open(path) as t:
                t.getmembers()
        return True
    except Exception as e:
        print(f"  Corrupt archive ({e}) — removing for clean re-download")
        os.remove(path)
        return False

def _extract(path, dest):
    print(f"  Extracting {os.path.basename(path)} ...")
    if path.endswith(".zip"):
        with zipfile.ZipFile(path) as z: z.extractall(dest)
    else:
        subprocess.run(["tar", "-xf", path, "-C", dest], check=True)


from huggingface_hub import hf_hub_download
import shutil

PRIVACY_DIR = f'{MODELS}/privacy'
BUFFALO_DIR = f'{PRIVACY_DIR}/buffalo_s'
DONE_FLAG   = f'{PRIVACY_DIR}/buffalo_done.flag'
os.makedirs(BUFFALO_DIR, exist_ok=True)

if os.path.exists(DONE_FLAG):
    print('✓ buffalo_s already downloaded')
else:
    # insightface downloads buffalo_s automatically on first prepare()
    # but we cache it to Drive so it survives Colab restarts.
    BUFFALO_URL = 'https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_s.zip'
    BUFFALO_ZIP = f'{PRIVACY_DIR}/buffalo_s.zip'
    ok = _safe_dl(BUFFALO_URL, BUFFALO_ZIP, min_mb=80)
    if not ok:
        print('⚠ Download failed — re-run to resume'); raise SystemExit
    if not _verify(BUFFALO_ZIP):
        print('Re-run to re-download corrupt archive'); raise SystemExit
    _extract(BUFFALO_ZIP, BUFFALO_DIR)
    open(DONE_FLAG, 'w').close()
    print('✅ buffalo_s ready')

# Tell insightface to use our cached copy
import insightface
face_app = insightface.app.FaceAnalysis(
    name='buffalo_s', root=PRIVACY_DIR,
    providers=['CUDAExecutionProvider','CPUExecutionProvider'])
face_app.prepare(ctx_id=0, det_size=(320, 320))
print('✅ FaceAnalysis ready')

## Cell 04-03 | Test Face Detection & Blur

In [ ]:
import cv2, numpy as np
from PIL import Image, ImageDraw
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def blur_faces(image_bgr, face_app, blur_strength=51):
    img_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    faces   = face_app.get(img_rgb)
    result  = image_bgr.copy()
    n_faces = len(faces)
    for face in faces:
        bbox = face.bbox.astype(int)
        x1, y1, x2, y2 = bbox[0], bbox[1], bbox[2], bbox[3]
        pad = 20
        x1 = max(0, x1-pad); y1 = max(0, y1-pad)
        x2 = min(image_bgr.shape[1], x2+pad); y2 = min(image_bgr.shape[0], y2+pad)
        roi = result[y1:y2, x1:x2]
        if roi.size > 0:
            result[y1:y2, x1:x2] = cv2.GaussianBlur(roi, (blur_strength, blur_strength), 0)
    return result, n_faces

# Generate a synthetic test image — no external URL dependency
# Draws a face-like oval on a plain background so the detector has something to try
pil_img = Image.new('RGB', (640, 480), color=(180, 160, 140))
draw = ImageDraw.Draw(pil_img)
draw.ellipse([220, 100, 420, 340], fill=(220, 185, 150), outline=(100, 80, 60), width=3)
draw.ellipse([260, 160, 300, 200], fill=(60, 40, 30))   # left eye
draw.ellipse([340, 160, 380, 200], fill=(60, 40, 30))   # right eye
draw.arc([280, 230, 370, 290], start=10, end=170, fill=(120, 60, 60), width=3)  # mouth
img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
print('Synthetic test image created (640x480)')

blurred, n = blur_faces(img, face_app)
print(f'Detected {n} face(s)')
os.makedirs(f'{BASE}/logs', exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(cv2.cvtColor(img,     cv2.COLOR_BGR2RGB)); axes[0].set_title('Original')
axes[1].imshow(cv2.cvtColor(blurred, cv2.COLOR_BGR2RGB)); axes[1].set_title('Privacy Protected')
plt.tight_layout(); plt.savefig(f'{BASE}/logs/privacy_test.png'); plt.close()
print(f'Saved: {BASE}/logs/privacy_test.png')
print('✅ Face blur pipeline working')


## Cell 04-04 | CRAFT Text Region Detection

In [ ]:
import subprocess, sys, os

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'easyocr'])
import easyocr, cv2, torch, numpy as np
from PIL import Image, ImageDraw

EASYOCR_DIR = f'{MODELS}/privacy/easyocr'
os.makedirs(EASYOCR_DIR, exist_ok=True)

print('Loading EasyOCR ...')
reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available(),
                         model_storage_directory=EASYOCR_DIR,
                         download_enabled=True, verbose=False)
print('EasyOCR ready')

# Synthesize a test image with readable text
pil_img = Image.new('RGB', (640, 320), color=(240, 240, 235))
draw = ImageDraw.Draw(pil_img)
draw.rectangle([20, 20, 620, 80],   fill=(255,255,255), outline=(200,200,200))
draw.rectangle([20, 100, 620, 160], fill=(255,255,255), outline=(200,200,200))
draw.text((30, 30),  'John Smith - 123 Main Street, New York', fill=(20,20,20))
draw.text((30, 110), 'PRIVATE PROPERTY - No Trespassing',      fill=(180,20,20))
img_bgr = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
print('Synthetic text image created (640x320)')

results = reader.detect(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB),
                         slope_ths=0.3, height_ths=1.0)
boxes = results[0] if results and results[0] else []
print(f'Text regions detected: {len(boxes)}')

result = img_bgr.copy()
for bbox in boxes:
    # EasyOCR detect() returns [x_min, x_max, y_min, y_max] but values
    # may be nested arrays in newer versions — flatten to be safe
    coords = np.array(bbox).flatten().astype(int)
    x_min, x_max, y_min, y_max = coords[0], coords[1], coords[2], coords[3]
    roi = result[y_min:y_max, x_min:x_max]
    if roi.size > 0:
        result[y_min:y_max, x_min:x_max] = cv2.GaussianBlur(roi, (31, 31), 0)

os.makedirs(f'{EXPORTS}', exist_ok=True)
Image.fromarray(cv2.cvtColor(result, cv2.COLOR_BGR2RGB)).save(
    f'{EXPORTS}/text_blur_test.jpg')
print(f'Text-blurred image saved: {EXPORTS}/text_blur_test.jpg')
print('EasyOCR text detection working.')

# Write done flag
from pathlib import Path
Path(EASYOCR_DIR, 'easyocr_done.flag').touch()
print('✅ EasyOCR done flag written')
